# Step 16: Autoencoder
The autoencoder squeezes 797 features into **32 numbers** and rebuilds them (no labels needed). Then the 32 numbers are used to predict engagement.
*Same as `src/16_autoencoder.py`.*

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: 9 conversations (English 027, 028, 030; French 001, 013, 021; German 070, 072, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # hide TensorFlow messages
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"    # same maths settings as the original script
if os.getcwd().endswith("notebooks"):
    os.chdir("..")                           # work from the project folder

In [ ]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

In [ ]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

In [ ]:
df = pd.read_parquet("data/features/noxi_engagement.parquet")

# the first 5 face columns are bookkeeping (frame number, face id, time, confidence, success)
bookkeeping = ["face_0", "face_1", "face_2", "face_3", "face_4"]
cols = []
for c in df.columns:
    if (c.startswith("voice_") or c.startswith("face_")) and c not in bookkeeping:
        cols.append(c)

X = df[cols].fillna(0).to_numpy("float32")      # features
y = df["engagement"].to_numpy("float32")        # labels
print(X.shape)

In [ ]:
# the 3 sessions of each language
english = sorted(df[df["language"] == "English"]["session"].unique())
french = sorted(df[df["language"] == "French"]["session"].unique())
german = sorted(df[df["language"] == "German"]["session"].unique())

# fold 1 tests on the 1st session of each language, fold 2 on the 2nd, fold 3 on the 3rd
folds = []
for k in range(3):
    folds.append([english[k], french[k], german[k]])
print(folds)

**Autoencoder:** encoder 797 → 256 → 32, decoder 32 → 256 → 797.

In [ ]:
def make_autoencoder():
    inp = layers.Input(shape=(797,))
    h = layers.Dense(256, activation="relu")(inp)
    code = layers.Dense(32, activation="relu")(h)
    h = layers.Dense(256, activation="relu")(code)
    out = layers.Dense(797)(h)
    autoencoder = keras.Model(inp, out)
    encoder = keras.Model(inp, code)
    return autoencoder, encoder

**Every fold:** (1) train the autoencoder to rebuild its input, (2) freeze the encoder and train a small head on the 32 numbers, (3) XGBoost on the same 32 numbers.

In [ ]:
scores = {"Autoencoder + dense head": [], "Autoencoder + XGBoost": []}

for test_sessions in folds:
    test = df["session"].isin(test_sessions).to_numpy()
    train = ~test
    scaler = StandardScaler().fit(X[train])
    X_train = np.clip(scaler.transform(X[train]), -5, 5)
    X_test = np.clip(scaler.transform(X[test]), -5, 5)

    # 1) autoencoder: input = target, no labels
    keras.utils.set_random_seed(42)
    autoencoder, encoder = make_autoencoder()
    autoencoder.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
    autoencoder.fit(X_train, X_train, epochs=30, batch_size=128, verbose=0)
    print("reconstruction error:", round(autoencoder.evaluate(X_test, X_test, verbose=0), 3))

    # 2) frozen encoder + small head
    encoder.trainable = False
    inp = layers.Input(shape=(797,))
    x = layers.Dense(64, activation="relu")(encoder(inp))
    x = layers.Dropout(0.1)(x)
    head = keras.Model(inp, layers.Dense(1)(x))
    head.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
    head.fit(X_train, y[train], epochs=30, batch_size=128, verbose=0)
    scores["Autoencoder + dense head"].append(ccc(y[test], head.predict(X_test, verbose=0).flatten()))

    # 3) XGBoost on the 32 numbers
    code_train = encoder.predict(X_train, verbose=0)
    code_test = encoder.predict(X_test, verbose=0)
    xgb = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                       colsample_bytree=0.8, tree_method="hist", random_state=42, n_jobs=-1)
    xgb.fit(code_train, y[train])
    scores["Autoencoder + XGBoost"].append(ccc(y[test], xgb.predict(code_test)))

In [ ]:
saved = pd.read_csv("reports/autoencoder_results.csv").groupby("model")["ccc"].mean()
for name in scores:
    print(name.ljust(26), "mean", round(np.mean(scores[name]), 3), " saved", round(saved[name], 3))